In [ ]:
"""
Created on Mon Mar 04 17:32 2024

Apply weights to long historical period

Author: @claraburgard

"""

In [ ]:
import xarray as xr
import numpy as np
from tqdm.notebook import tqdm
import seaborn as sns
import multimelt.useful_functions as uf
import random

In [ ]:
sns.set_context('paper')

In [ ]:
%matplotlib qt5

READ IN DATA

In [ ]:
home_path = '/bettik/burgardc/'
plot_path = '/bettik/burgardc/PLOTS/summer_paper_plots/'
outputpath_GL = '/bettik/burgardc/DATA/SUMMER_PAPER/processed/GL_FLUX/'
inputpath_weights = '/bettik/burgardc/DATA/SUMMER_PAPER/processed/ANALYSIS/'
inputpath_atmo = '/bettik/burgardc/DATA/SUMMER_PAPER/raw/TS_SMB_DATA/out/'


In [ ]:
inputpath_mask = home_path+'/DATA/SUMMER_PAPER/interim/ANTARCTICA_IS_MASKS/BedMachine_4km/'
file_isf_orig = xr.open_dataset(inputpath_mask+'BedMachinev2_4km_isf_masks_and_info_and_distance_oneFRIS.nc')
nonnan_Nisf = file_isf_orig['Nisf'].where(np.isfinite(file_isf_orig['front_bot_depth_max']), drop=True).astype(int)
file_isf_nonnan = file_isf_orig.sel(Nisf=nonnan_Nisf)
rignot_isf = file_isf_nonnan.Nisf.where(np.isfinite(file_isf_nonnan['isf_area_rignot']), drop=True)
file_isf = file_isf_nonnan.sel(Nisf=rignot_isf)

In [ ]:
sorted_isf_rignot = [11,69,43,28,12,57,
                     70,44,29,13,58,71,45,30,14,
                     59,72,46,
                     31,
                     15,61,73,47,32,16,48,33,17,62,49,34,18,63,74,
                     50,35,19,64,
                     10,
                     36,20,65,51,37,
                     22,38,52,23,66,53,39,24,
                     67,40,54,75,25,41,
                     26,42,55,68,60,27]

In [ ]:
param_classic_list = ['linear_local',
              'quadratic_local','quadratic_local_locslope',
              'quadratic_mixed_mean','quadratic_mixed_locslope',
              'lazero19',
              'boxes_4_pismyes_picopno']

param_NN_list = ['NNsmall']

CALVING

In [ ]:
inputpath_davison = home_path+'/DATA/SUMMER_PAPER/raw/'
davison_file = xr.open_dataset(inputpath_davison + 'steadystate_davison23.nc')
calving_flux = davison_file['calving_obs']

ABUMIP GL FLUX

In [ ]:
GL_flux = xr.open_dataset(outputpath_GL + 'all_GL_fluxes_varying_m.nc')

In [ ]:
GL_flux_ag = xr.Dataset()
GL_flux_ag['flux_Gt_ref'] = xr.concat([GL_flux['flux_Gt_ref_m1'].assign_coords({'m': 1}),
                                       GL_flux['flux_Gt_ref_m3'].assign_coords({'m': 3}),
                                       GL_flux['flux_Gt_ref_m5'].assign_coords({'m': 5})], dim='m')
GL_flux_ag['flux_Gt_ABUMIP'] = xr.concat([GL_flux['flux_Gt_ABUMIP_m1'].assign_coords({'m': 1}),
                                       GL_flux['flux_Gt_ABUMIP_m3'].assign_coords({'m': 3}),
                                       GL_flux['flux_Gt_ABUMIP_m5'].assign_coords({'m': 5})], dim='m')

GL_flux_ABUMIP = GL_flux_ag['flux_Gt_ABUMIP']

SMB

In [ ]:
def start_end_year(mod, scen):
    if mod in ['CNRM-CM6-1','CNRM-ESM2-1']:
        ens_run = 'r1i1p1f2'
        to2300 = False
    elif mod in ['GISS-E2-1-H']:
        ens_run = 'r1i1p1f2'
        to2300 = True
    elif mod in ['ACCESS-CM2','ACCESS-ESM1-5','CESM2-WACCM','CanESM5','IPSL-CM6A-LR','MRI-ESM2-0']:
        ens_run = 'r1i1p1f1'
        to2300 = True
    elif mod in ['MPI-ESM1-2-HR','GFDL-CM4','GFDL-ESM4']:
        ens_run = 'r1i1p1f1'
        to2300 = False
    elif mod == 'UKESM1-0-LL':
        ens_run = 'r4i1p1f2'
        to2300 = True     
    elif mod == 'CESM2':
        ens_run = 'r11i1p1f1'
        to2300 = False        


    if scenario == 'historical':
        yystart = 1850 #1850
        yyend = 2014
    elif scenario == 'ssp245':
        yystart = 2015
        yyend = 2100  
    else:
        if to2300:
            yystart = 2015
            yyend = 2299
        else:
            yystart = 2015
            yyend = 2100  
    
    return ens_run, yystart, yyend

In [ ]:
melt_atmo_list2 = []
inputpath_atmo_Chris = '/bettik/burgardc/DATA/SUMMER_PAPER/raw/TS_SMB_DATA/out2/' 
inputpath_atmo_Nico = '/bettik/burgardc/DATA/SUMMER_PAPER/interim/SMB_EMULATED/'

    
melt_atmo3_list = []
for scenario in ['historical','ssp126','ssp245','ssp585']: #

    melt_atmo_list = []
    for mod in ['CESM2','IPSL-CM6A-LR','MPI-ESM1-2-HR','UKESM1-0-LL']: #'CNRM-CM6-1',
        
        _, yystart, yyend = start_end_year(mod, scenario)
        #print(mod,yystart,yyend)
        
        if (scenario == 'historical') and (yystart == 1850):
            
            melt_atmo_1850 = xr.open_dataset(inputpath_atmo_Nico+mod+'_SMB_'+scenario+'-1850-1979.nc').sel(Nisf=rignot_isf)
            
            if (os.path.exists(inputpath_atmo_Chris+mod+'_SMB_'+scenario+'-1980-2014.nc')):
                melt_atmo = xr.open_dataset(inputpath_atmo_Chris+mod+'_SMB_'+scenario+'-1980-2014.nc').sel(Nisf=rignot_isf)
                melt_atmo['time'] = melt_atmo['time'].dt.year
            else:
                melt_atmo = xr.open_dataset(inputpath_atmo_Nico+mod+'_SMB_'+scenario+'-1980-2014.nc').sel(Nisf=rignot_isf)
                
            melt_atmo_list.append(xr.concat([melt_atmo_1850['SMB'],melt_atmo['SMB']], dim='time').assign_coords({'model': mod}))

        else:
            
            if os.path.exists(inputpath_atmo_Chris+mod+'_SMB_'+scenario+'-'+str(yystart)+'-'+str(yyend)+'.nc'):
                melt_atmo = xr.open_dataset(inputpath_atmo_Chris+mod+'_SMB_'+scenario+'-'+str(yystart)+'-'+str(yyend)+'.nc').sel(Nisf=rignot_isf)
                melt_atmo['time'] = melt_atmo['time'].dt.year
                melt_atmo_list.append(melt_atmo['SMB'].assign_coords({'model': mod}))
            else:
                melt_atmo = xr.open_dataset(inputpath_atmo_Nico+mod+'_SMB_'+scenario+'-'+str(yystart)+'-'+str(yyend)+'.nc').sel(Nisf=rignot_isf)
                melt_atmo_list.append(melt_atmo['SMB'].assign_coords({'model': mod}))


    for mod in ['ACCESS-ESM1-5','ACCESS-CM2','CESM2-WACCM','CNRM-CM6-1','CNRM-ESM2-1',
                'CanESM5','GFDL-CM4','GFDL-ESM4', 
               'MRI-ESM2-0']: #'GISS-E2-1-H',

        _, yystart, yyend = start_end_year(mod, scenario)
        #print(mod,yystart,yyend)
        
        if mod == 'GFDL-CM4' and scenario == 'ssp126':
            melt_atmo = melt_atmo * np.nan
        
        else:
            
            if (scenario == 'historical') and (yystart == 1850):
                    
                melt_atmo_1850 = xr.open_dataset(inputpath_atmo_Nico+mod+'_SMB_'+scenario+'-1850-1979.nc').sel(Nisf=rignot_isf)
                melt_atmo = xr.open_dataset(inputpath_atmo_Nico+mod+'_SMB_'+scenario+'-1980-2014.nc').sel(Nisf=rignot_isf)
                melt_atmo_list.append(xr.concat([melt_atmo_1850['SMB'],melt_atmo['SMB']], dim='time').assign_coords({'model': mod}))
            
            else:
            
                melt_atmo = xr.open_dataset(inputpath_atmo_Nico+mod+'_SMB_'+scenario+'-'+str(yystart)+'-'+str(yyend)+'.nc').sel(Nisf=rignot_isf)
                melt_atmo_list.append(melt_atmo['SMB'].assign_coords({'model': mod}))

    melt_atmo_scenario = xr.concat(melt_atmo_list, dim='model')
    melt_atmo3_list.append(melt_atmo_scenario.assign_coords({'scenario': scenario}))
melt_atmo_all0 = xr.concat(melt_atmo3_list, dim='scenario').sel(Nisf=rignot_isf)
#melt_atmo_all = xr.concat(melt_atmo_list2, dim='model')


In [ ]:
concat_melt_atmo_list = []

for scen in ['ssp126','ssp245','ssp585']:
    concat_melt_atmo = xr.concat([melt_atmo_all0.sel(time=range(1850,2015),scenario='historical').drop('scenario'), melt_atmo_all0.sel(time=range(2015,2300),scenario=scen).drop('scenario')], dim='time')
    concat_melt_atmo_list.append(concat_melt_atmo.assign_coords({'scenario': scen}))
melt_atmo_all = xr.concat(concat_melt_atmo_list, dim='scenario')

##### OCEAN

In [ ]:
Gt_list_models = []
box1_list_models = []

for mod in tqdm(['ACCESS-ESM1-5','ACCESS-CM2','CESM2','CESM2-WACCM','CNRM-CM6-1','CNRM-ESM2-1',
            'CanESM5','GFDL-CM4','GFDL-ESM4', 'IPSL-CM6A-LR',
           'MPI-ESM1-2-HR','MRI-ESM2-0','UKESM1-0-LL']): #'GISS-E2-1-H',
    
    outputpath_melt = '/bettik/burgardc/DATA/SUMMER_PAPER/processed/OCEAN_MELT_RATE_CMIP/'+mod+'/'

    melt1D_list1 = []

    for mparam in param_classic_list:

        melt1D_scenario_list = []
        for scenario in ['historical','ssp126','ssp245','ssp585']:
            if mod == 'GFDL-CM4' and scenario == 'ssp126':
                melt_1D = melt_1D * np.nan 
            else:
                melt_1D = xr.open_dataset(outputpath_melt + 'eval_metrics_1D_'+mparam+'_'+scenario+'_oneFRIS_anoISMIP.nc')
                melt1D_scenario_list.append(melt_1D.assign_coords({'scenario': scenario}))
        melt1D_scenario = xr.concat(melt1D_scenario_list, dim='scenario')
        melt1D_list1.append(melt1D_scenario.assign_coords({'param':mparam}))
    melt1D_classic = xr.concat(melt1D_list1, dim='param')       
    Gt_classic = melt1D_classic['melt_1D_Gt_per_y']
    box1_classic = melt1D_classic['melt_1D_mean_myr_box1']
    
    melt1D_list2 = []
    for mparam in param_NN_list:

        melt1D_scenario_list = []
        for scenario in ['historical','ssp126','ssp245','ssp585']:
            if mod == 'GFDL-CM4' and scenario == 'ssp126':
                melt_1D = melt_1D * np.nan 
            else:
                melt_1D = xr.open_dataset(outputpath_melt + 'evalmetrics_1D_'+mparam+'_'+scenario+'_anoISMIP.nc')
                melt1D_scenario_list.append(melt_1D.assign_coords({'scenario': scenario}))
        melt1D_scenario = xr.concat(melt1D_scenario_list, dim='scenario')
        melt1D_list2.append(melt1D_scenario.assign_coords({'param':mparam}))
    melt1D_NN = xr.concat(melt1D_list2, dim='param')   
    Gt_NN = melt1D_NN['predicted_melt'].sel(metrics='Gt')
    box1_NN = melt1D_NN['predicted_melt'].sel(metrics='box1')
    
    Gt_all = xr.concat([Gt_classic, Gt_NN], dim='param')
    box1_all = xr.concat([box1_classic, box1_NN], dim='param')
    
    Gt_list_models.append(Gt_all.assign_coords({'model':mod}))
    box1_list_models.append(box1_all.assign_coords({'model':mod}))

Gt_all_mod_anoISMIP = xr.concat(Gt_list_models, dim='model')
box1_all_mod_anoISMIP = xr.concat(box1_list_models, dim='model')


    
    

In [ ]:
Gt_list_models = []
box1_list_models = []

for mod in tqdm(['ACCESS-ESM1-5','ACCESS-CM2','CESM2','CESM2-WACCM','CNRM-CM6-1','CNRM-ESM2-1',
            'CanESM5','GFDL-CM4','GFDL-ESM4', 'IPSL-CM6A-LR',
           'MPI-ESM1-2-HR','MRI-ESM2-0','UKESM1-0-LL']): #'GISS-E2-1-H',
    
    outputpath_melt = '/bettik/burgardc/DATA/SUMMER_PAPER/processed/OCEAN_MELT_RATE_CMIP/'+mod+'/'

    melt1D_list1 = []

    for mparam in param_classic_list:

        melt1D_scenario_list = []
        for scenario in ['historical','ssp126','ssp245','ssp585']:
            if mod == 'GFDL-CM4' and scenario == 'ssp126':
                melt_1D = melt_1D * np.nan 
            else:
                melt_1D = xr.open_dataset(outputpath_melt + 'eval_metrics_1D_'+mparam+'_'+scenario+'_oneFRIS_anoNEMO.nc')
                melt1D_scenario_list.append(melt_1D.assign_coords({'scenario': scenario}))
        melt1D_scenario = xr.concat(melt1D_scenario_list, dim='scenario')
        melt1D_list1.append(melt1D_scenario.assign_coords({'param':mparam}))
    melt1D_classic = xr.concat(melt1D_list1, dim='param')       
    Gt_classic = melt1D_classic['melt_1D_Gt_per_y']
    box1_classic = melt1D_classic['melt_1D_mean_myr_box1']
    
    melt1D_list2 = []
    for mparam in param_NN_list:

        melt1D_scenario_list = []
        for scenario in ['historical','ssp126','ssp245','ssp585']:
            if mod == 'GFDL-CM4' and scenario == 'ssp126':
                melt_1D = melt_1D * np.nan 
            else:
                melt_1D = xr.open_dataset(outputpath_melt + 'evalmetrics_1D_'+mparam+'_'+scenario+'_anoNEMO.nc')
                melt1D_scenario_list.append(melt_1D.assign_coords({'scenario': scenario}))
        melt1D_scenario = xr.concat(melt1D_scenario_list, dim='scenario')
        melt1D_list2.append(melt1D_scenario.assign_coords({'param':mparam}))
    melt1D_NN = xr.concat(melt1D_list2, dim='param')   
    Gt_NN = melt1D_NN['predicted_melt'].sel(metrics='Gt')
    box1_NN = melt1D_NN['predicted_melt'].sel(metrics='box1')
    
    Gt_all = xr.concat([Gt_classic, Gt_NN], dim='param')
    box1_all = xr.concat([box1_classic, box1_NN], dim='param')
    
    Gt_list_models.append(Gt_all.assign_coords({'model':mod}))
    box1_list_models.append(box1_all.assign_coords({'model':mod}))

Gt_all_mod_anoNEMO = xr.concat(Gt_list_models, dim='model')
box1_all_mod_anoNEMO = xr.concat(box1_list_models, dim='model')


    
    

In [ ]:
ano_choice_file = xr.open_dataset(inputpath_weights + 'ano_choice_NEMOorISMIP_withoutGISS.nc')


bmb_combined_list = []
for kisf in tqdm(file_isf.Nisf):
    if ano_choice_file['ano_choice'].sel(Nisf=kisf).values == 'NEMO':
        bmb_combined_list.append(Gt_all_mod_anoNEMO.sel(Nisf=kisf))
    elif ano_choice_file['ano_choice'].sel(Nisf=kisf).values == 'ISMIP':
        bmb_combined_list.append(Gt_all_mod_anoISMIP.sel(Nisf=kisf))
bmb_combined0 = xr.concat(bmb_combined_list, dim='Nisf')

concat_bmb_list = []

for scen in ['ssp126','ssp245','ssp585']:
    concat_bmb = xr.concat([bmb_combined0.sel(time=range(1850,2015),scenario='historical').drop('scenario'), bmb_combined0.sel(time=range(2015,2300),scenario=scen).drop('scenario')], dim='time')
    concat_bmb_list.append(concat_bmb.assign_coords({'scenario': scen}))
bmb_combined = xr.concat(concat_bmb_list, dim='scenario')

In [ ]:
bmb_alldim, GL_flux_ABUMIP_alldim  = xr.broadcast(bmb_combined, GL_flux_ABUMIP)
_, smb_alldim  = xr.broadcast(bmb_alldim, melt_atmo_all)
_, calv_alldim = xr.broadcast(bmb_alldim, davison_file['calving_obs'])

MASS BALANCE

In [ ]:
weight_file = xr.open_dataset(inputpath_weights + 'bayesian_weights_davison_varying_combined_withoutGISS.nc')
weight_2300_file = xr.open_dataset(inputpath_weights + 'bayesian_weights_davison_varying_combined_2300_withoutGISS.nc')

In [ ]:
mass_balance_simu = bmb_combined - melt_atmo_all + davison_file['calving_obs'] - GL_flux_ag['flux_Gt_ABUMIP']

In [ ]:
mass_balance_simu_2300 = mass_balance_simu.sel(model=['ACCESS-CM2','ACCESS-ESM1-5','CanESM5',
                                                        'CESM2-WACCM', 'IPSL-CM6A-LR',
                                                        'MRI-ESM2-0','UKESM1-0-LL']) #'GISS-E2-1-H',


WRITE INTO NETCDF FILE FOR FURTHER ANALYSIS IN OTHER NOTEBOOK

In [ ]:
file_viability_info = xr.Dataset()
file_viability_info['BMB'] = bmb_alldim
file_viability_info['SMB'] = smb_alldim
file_viability_info['ABUMIP'] = GL_flux_ABUMIP_alldim
file_viability_info['CALVING'] = calv_alldim
file_viability_info['MASS_BALANCE'] = mass_balance_simu
file_viability_info.to_netcdf(inputpath_weights + 'all_fluxes_br_withoutGISS.nc')